# Train CaiTien_HGT on Google Colab

Recommended before running:
- `Runtime -> Change runtime type -> T4 GPU`
- If the setup cell falls back to CPU DGL, try another Colab runtime whose CUDA maps to `cu118`, `cu121`, or `cu124`.

This notebook clones the repo, installs dependencies, runs a smoke test, then shows the command for full training.


In [ ]:
%cd /content
!rm -rf CaiTien_HGT
!git clone https://github.com/DucTri2207/CaiTien_HGT.git
%cd /content/CaiTien_HGT


In [ ]:
import os
import sys
import subprocess

os.environ['DGLBACKEND'] = 'pytorch'

def run(*args):
    print('$', ' '.join(args))
    subprocess.check_call(list(args))

run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt')

import torch

cuda_tag = None
cuda_version = torch.version.cuda or ''
if torch.cuda.is_available():
    if cuda_version.startswith('12.4'):
        cuda_tag = 'cu124'
    elif cuda_version.startswith('12.1'):
        cuda_tag = 'cu121'
    elif cuda_version.startswith('11.8'):
        cuda_tag = 'cu118'

if cuda_tag is not None:
    dgl_repo = f'https://data.dgl.ai/wheels/{cuda_tag}/repo.html'
    run(sys.executable, '-m', 'pip', 'install', '-q', 'dgl', '-f', dgl_repo)
    os.environ['AMDGT_DEVICE'] = 'cuda'
else:
    run(sys.executable, '-m', 'pip', 'install', '-q', 'dgl')
    os.environ['AMDGT_DEVICE'] = 'cpu'

print('torch', torch.__version__)
print('torch.cuda.is_available()', torch.cuda.is_available())
print('torch.version.cuda', torch.version.cuda)
print('AMDGT_DEVICE', os.environ['AMDGT_DEVICE'])
if os.environ['AMDGT_DEVICE'] == 'cpu':
    print('GPU DGL was not auto-selected. Smoke test and training will run on CPU.')


In [ ]:
import os
import sys
import subprocess
import torch
import dgl
import pandas
import sklearn
import networkx

print('python', sys.version)
print('torch', torch.__version__)
print('dgl', dgl.__version__)
print('pandas', pandas.__version__)
print('sklearn', sklearn.__version__)
print('networkx', networkx.__version__)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
import os
import sys
import subprocess

env = dict(os.environ)
subprocess.check_call([
    sys.executable,
    'train_DDA.py',
    '--dataset', 'C-dataset',
    '--epochs', '1',
    '--k_fold', '2',
], env=env)


In [ ]:
# Full training example
import os
import sys
import subprocess

env = dict(os.environ)
subprocess.check_call([
    sys.executable,
    'train_DDA.py',
    '--dataset', 'C-dataset',
    '--epochs', '1000',
    '--k_fold', '10',
], env=env)
